# SLAM-ASR: Google Drive Checkpoint Değerlendirme Aracı (Mozilla Common Voice)
### Eğitilmiş Tüm Modellerin Common Voice (CV-TR) Üzerinde Hızlı Çapraz Korpus Testi

Bu notebook, Google Drive'ınızda (`/content/drive/MyDrive/checkpoints/`) kayıtlı olan tüm SLAM-ASR Türkçe modellerini (`t0`, `t1-mlp-lora`, `t1-lora`, `t1-tail-aug-lora`, `t1-tail-aug`, `t3-ng-ta-gormez` vb.) **yeniden eğitime gerek kalmadan** Mozilla Common Voice Türkçe test seti üzerinde hızlıca değerlendirmek için tasarlanmıştır.

* **Yeniden Eğitim:** Gerekmez (Ağırlıklar doğrudan Drive'dan yüklenir).
* **Değerlendirme Süresi:** Model başına yaklaşık ~6-7 dakika (GPU ile).
* **Eski Klasörü:** Drive'daki `Eski/` arşiv klasörü otomatik olarak göz ardı edilir.
* **Otomatik Kayıt:** Hesaplanan `eval_results_cv_generalization.jsonl` sonuçları doğrudan Google Drive'daki ilgili model klasörüne kalıcı olarak kaydedilir.


In [1]:
# 1. GPU Kontrolü ve Google Drive Bağlantısı
!nvidia-smi

import os
from google.colab import drive

if not os.path.exists("/content/drive/MyDrive"):
    drive.mount("/content/drive")
    print("[*] Google Drive başarıyla bağlandı.")
else:
    print("[*] Google Drive zaten bağlı.")


Fri Oct 23 15:31:38 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   58C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# 2. Çalışma Dizinini Hazırlama ve Bağımlılıkları Yükleme
import os

if not os.path.exists("slam-asr-reimplement"):
    !git clone https://github.com/gokhanersoy/slam-asr-reimplement.git

# Tüm projektör mimarilerini destekleyen t5 dizinine geçin
%cd /content/slam-asr-reimplement/tr/tsc/t5-ng-ta-gormez-tail-aug-lora

!pip install -q -r requirements.txt
!pip install -q huggingface_hub soundfile torchaudio bitsandbytes accelerate datacollective mutagen peft


Cloning into 'slam-asr-reimplement'...
remote: Enumerating objects: 1816, done.
remote: Counting objects: 100% (667/667), done.
remote: Compressing objects: 100% (474/474), done.
remote: Total 1816 (delta 466), reused 384 (delta 192), pack-reused 1149 (from 1)
Receiving objects: 100% (1816/1816), 15.87 MiB | 7.03 MiB/s, done.
Resolving deltas: 100% (869/869), done.
/content/slam-asr-reimplement/tr/tsc/t5-ng-ta-gormez-tail-aug-lora
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 62.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 126.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 118.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.3/71.3 kB 760.2 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 195.7/195.7 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that 

In [ ]:
# 3. Mozilla Common Voice Test Verisini Hazırlama (Tek Seferlik)
import os
import json
from google.colab import userdata

cv_test_manifest = "data/cv/test.jsonl"

def is_valid_manifest(path):
    if not path or not os.path.exists(path):
        return False
    try:
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                if not line.strip():
                    continue
                data = json.loads(line)
                audio_path = data.get("audio_filepath", "")
                return bool(audio_path and os.path.exists(audio_path))
    except Exception:
        return False
    return False

if not is_valid_manifest(cv_test_manifest):
    mdc_api_key = os.environ.get("MDC_API_KEY")
    if not mdc_api_key:
        try:
            mdc_api_key = userdata.get("MDC_API_KEY")
        except Exception:
            mdc_api_key = input("Lütfen Mozilla Data Collective API anahtarınızı (MDC_API_KEY) girin: ")

    print("[*] Common Voice test verisi indiriliyor ve hazırlanıyor...")
    !python scripts/prepare_common_voice.py \
        --download \
        --api_key "{mdc_api_key}" \
        --output_dir data/cv
else:
    print(f"[*] Common Voice test manifestosu ve ses dosyaları hazır: {cv_test_manifest}")


[*] Common Voice test verisi indiriliyor ve hazırlanıyor...
[*] Connecting to Mozilla Data Collective REST API...
[*] Requesting presigned download URL from Mozilla Data Collective API...
    URL: https://mozilladatacollective.com/api/datasets/cmu5wkah500c2o10719lem8m8/download
[*] Downloading dataset to: /content/slam-asr-reimplement/tr/tsc/t5-ng-ta-gormez-tail-aug-lora/data/cv/common-voice-scripted-speech-27-0-turkis-b14760fe.tar.gz
    Progress: 0% (8.0 MB / 2853.8 MB)
    Progress: 5% (144.0 MB / 2853.8 MB)
    Progress: 10% (288.0 MB / 2853.8 MB)
    Progress: 15% (432.0 MB / 2853.8 MB)
    Progress: 20% (576.0 MB / 2853.8 MB)
    Progress: 25% (720.0 MB / 2853.8 MB)
    Progress: 30% (864.0 MB / 2853.8 MB)
    Progress: 35% (1000.0 MB / 2853.8 MB)
    Progress: 40% (1144.0 MB / 2853.8 MB)
    Progress: 45% (1288.0 MB / 2853.8 MB)
    Progress: 50% (1432.0 MB / 2853.8 MB)
    Progress: 55% (1576.0 MB / 2853.8 MB)
    Progress: 60% (1720.0 MB / 2853.8 MB)
    Progress: 65% (1856.0 

In [4]:
# 4. Google Drive'daki Mevcut TSC Kontrol Noktalarını Tarama ('Eski' Klasörü Hariç)
import glob
import os
import re
import json

checkpoints_root = "/content/drive/MyDrive/checkpoints"

def extract_model_tag(folder_name):
    # e.g. 20261018_170924_slam_asr_tr_tsc_t0 -> t0
    # e.g. 20261021_045752_slam_asr_tr_tsc_t1_mlp_lora -> t1-mlp-lora
    # e.g. 20261018_170924_slam_asr_tr_tsc -> t0
    m = re.search(r"slam_asr_tr_tsc_?(.*)", folder_name)
    if m and m.group(1):
        tag = m.group(1).replace("_", "-").strip("-")
        return tag if tag else "t0"
    return "t0"

# 1. Tüm adayları bul ve 'Eski' / 'eski' klasörünü otomatik hariç tut
raw_checkpoints = glob.glob(f"{checkpoints_root}/*slam_asr_tr_tsc*/best_checkpoint")
raw_checkpoints += glob.glob(f"{checkpoints_root}/*/*slam_asr_tr_tsc*/best_checkpoint")

all_tsc_checkpoints = []
for p in raw_checkpoints:
    p_norm = p.replace("\\", "/")
    if "/eski/" in p_norm.lower():
        continue
    if p not in all_tsc_checkpoints:
        all_tsc_checkpoints.append(p)

all_tsc_checkpoints = sorted(
    all_tsc_checkpoints,
    key=lambda p: (re.search(r"(\d{8}_\d{6})", p).group(1) if re.search(r"(\d{8}_\d{6})", p) else "", p)
)

print(f"[*] Google Drive'da aktif {len(all_tsc_checkpoints)} adet TSC kontrol noktası tespit edildi ('Eski' hariç):\n")
print(f"{'No':<3} | {'Model Etiketi':<22} | {'Model Klasörü':<45} | {'CV Durumu':<14} | {'Mevcut WER / CER'}")
print("-" * 115)

def get_existing_metrics(folder_dir):
    summary_file = os.path.join(folder_dir, "eval_results_cv_summary.json")
    if os.path.exists(summary_file):
        try:
            with open(summary_file, "r", encoding="utf-8") as f:
                d = json.load(f)
                w, c = d.get("wer"), d.get("cer")
                if w is not None and c is not None:
                    return float(w), float(c)
        except Exception:
            pass
    eval_file = os.path.join(folder_dir, "eval_results_cv_generalization.jsonl")
    if not os.path.exists(eval_file) or os.path.getsize(eval_file) == 0:
        return None, None
    hyps, refs = [], []
    try:
        with open(eval_file, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                res = json.loads(line)
                if "metrics" in res or ("wer" in res and "hypothesis" not in res):
                    wer = res.get("metrics", {}).get("wer") or res.get("wer")
                    cer = res.get("metrics", {}).get("cer") or res.get("cer")
                    if wer is not None and cer is not None:
                        w = wer * 100 if wer <= 1.0 else wer
                        c = cer * 100 if cer <= 1.0 else cer
                        return float(w), float(c)
                hyps.append(res.get("hypothesis", ""))
                refs.append(res.get("reference", ""))
        if hyps and refs:
            from slam_asr.utils.metrics import compute_wer, compute_cer
            wer = compute_wer(hyps, refs, language="tr")
            cer = compute_cer(hyps, refs, language="tr")
            w = wer * 100 if wer <= 1.0 else wer
            c = cer * 100 if cer <= 1.0 else cer
            try:
                with open(summary_file, "w", encoding="utf-8") as sf:
                    json.dump({"wer": round(w, 2), "cer": round(c, 2), "samples": len(hyps)}, sf, indent=2)
            except Exception:
                pass
            return float(w), float(c)
    except Exception:
        pass
    return None, None

checkpoint_status = []
for idx, ckpt in enumerate(all_tsc_checkpoints, 1):
    folder_dir = os.path.dirname(ckpt.rstrip("/"))
    folder_name = os.path.basename(folder_dir)
    model_tag = extract_model_tag(folder_name)
    eval_file = os.path.join(folder_dir, "eval_results_cv_generalization.jsonl")

    w, c = get_existing_metrics(folder_dir)
    if w is not None and c is not None:
        status = "TAMAMLANDI"
        metrics = f"WER: {w:.2f}%, CER: {c:.2f}%"
    elif os.path.exists(eval_file) and os.path.getsize(eval_file) > 0:
        status = "TAMAMLANDI"
        metrics = "JSONL Mevcut"
    else:
        status = "EKSİK (Test Edilecek)"
        metrics = "-"

    checkpoint_status.append({
        "index": idx,
        "model_tag": model_tag,
        "folder_name": folder_name,
        "folder_dir": folder_dir,
        "best_checkpoint": ckpt,
        "status": status,
        "metrics": metrics
    })
    print(f"{idx:<3} | {model_tag:<22} | {folder_name[:45]:<45} | {status:<14} | {metrics}")


[*] Google Drive'da aktif 16 adet TSC kontrol noktası tespit edildi ('Eski' hariç):

No  | Model Etiketi          | Model Klasörü                                 | CV Durumu      | Mevcut WER / CER
-------------------------------------------------------------------------------------------------------------------
1   | t0                     | 20261016_044206_slam_asr_tr_tsc_t0            | EKSİK (Test Edilecek) | -
2   | t1-pa                  | 20261018_005144_slam_asr_tr_tsc_t1_pa         | EKSİK (Test Edilecek) | -
3   | t1-ma                  | 20261018_010517_slam_asr_tr_tsc_t1_ma         | EKSİK (Test Edilecek) | -
4   | t1-sa                  | 20261018_090616_slam_asr_tr_tsc_t1_sa         | EKSİK (Test Edilecek) | -
5   | t1-ng                  | 20261018_091459_slam_asr_tr_tsc_t1_ng         | EKSİK (Test Edilecek) | -
6   | t2-ng-ta               | 20261018_175735_slam_asr_tr_tsc_t2_ng_ta      | EKSİK (Test Edilecek) | -
7   | t3-ng-ta-gormez        | 20261019_182924_slam_asr_

In [ ]:
# 5. Eksik Olan Modelleri Sırayla Common Voice Üzerinde Değerlendirme
import subprocess

# Tüm eksikleri otomatik test etmek için True bırakın.
# (Belirli modelleri seçmek isterseniz: target_indices = [3, 5] gibi liste verebilirsiniz)
run_all_missing = True

#target_indices = None
# Sadece eksik ve en kritik 5 modeli seçmek için:
target_indices = [7, 12, 13, 14, 15]
# 7:  t3-ng-ta-gormez
# 12: t1-tail-aug
# 13: t1-lora
# 14: t1-mlp-lora
# 15: t1-tail-aug-lora

models_to_evaluate = []
for item in checkpoint_status:
    if target_indices is not None:
        if item["index"] in target_indices:
            models_to_evaluate.append(item)
    elif run_all_missing and "EKSİK" in item["status"]:
        models_to_evaluate.append(item)

print(f"[*] Toplam {len(models_to_evaluate)} model değerlendirilecek:
")
for m in models_to_evaluate:
    print(f"  • [{m['index']}] {m['model_tag']:<20} ({m['folder_name']})")

print("
" + "=" * 75)

for m in models_to_evaluate:
    folder_dir = m["folder_dir"]
    ckpt_path = m["best_checkpoint"]
    config_path = os.path.join(folder_dir, "hardware_adaptive_turkish.yaml")
    if not os.path.exists(config_path):
        config_path = "config/hardware_adaptive_turkish.yaml"

    output_file = os.path.join(folder_dir, "eval_results_cv_generalization.jsonl")

    print(f"
[*] Değerlendirme Başlatılıyor: [{m['model_tag']}] {m['folder_name']}")
    print(f"    Checkpoint:  {ckpt_path}")
    print(f"    Config:      {config_path}")
    print(f"    Hedef Çıktı: {output_file}")

    cmd = [
        "python", "evaluate.py",
        "--config", config_path,
        "--checkpoint", ckpt_path,
        "--manifest", "data/cv/test.jsonl",
        "--output_file", "temp_eval_cv.jsonl",
        "--language", "tr",
        "--beam_size", "4"
    ]

    ret = subprocess.run(cmd)
    if ret.returncode == 0 and os.path.exists("temp_eval_cv.jsonl"):
        !cp temp_eval_cv.jsonl "{output_file}"
        print(f"[✓] Başarılı! Sonuç Drive'a kaydedildi: {output_file}")
        w, c = get_existing_metrics(folder_dir)
        if w is not None and c is not None:
            print(f"    -> [SKOR] WER: {w:.2f}%, CER: {c:.2f}%
")
    else:
        print(f"[!] {m['folder_name']} değerlendirmesi sırasında bir sorun oluştu.")

print("
" + "=" * 75)
print("[*] Tüm seçilen modellerin değerlendirmesi tamamlandı!")

[*] Toplam 5 model değerlendirilecek:

  • [7] t3-ng-ta-gormez      (20261019_182924_slam_asr_tr_tsc_t3_ng_ta_gormez)
  • [12] t1-tail-aug          (20261021_122138_slam_asr_tr_tsc_t1_tail_aug)
  • [13] t1-lora              (20261021_123811_slam_asr_tr_tsc_t1_lora)
  • [14] t1-mlp-lora          (20261021_124032_slam_asr_tr_tsc_t1_mlp_lora)
  • [15] t1-tail-aug-lora     (20261022_031103_slam_asr_tr_tsc_t1_tail_aug_lora)


[*] Değerlendirme Başlatılıyor: [t3-ng-ta-gormez] 20261019_182924_slam_asr_tr_tsc_t3_ng_ta_gormez
    Checkpoint:  /content/drive/MyDrive/checkpoints/20261019_182924_slam_asr_tr_tsc_t3_ng_ta_gormez/best_checkpoint
    Config:      /content/drive/MyDrive/checkpoints/20261019_182924_slam_asr_tr_tsc_t3_ng_ta_gormez/hardware_adaptive_turkish.yaml
    Hedef Çıktı: /content/drive/MyDrive/checkpoints/20261019_182924_slam_asr_tr_tsc_t3_ng_ta_gormez/eval_results_cv_generalization.jsonl
[✓] Başarılı! Sonuç Drive'a kaydedildi: /content/drive/MyDrive/checkpoints/20261019_182924_sl

In [ ]:
# 6. Nihai Değerlendirme Tablosunu Yazdırma (Master Results Summary)
import json
import os
import sys

for cand in ["/content/slam-asr-reimplement/tr/tsc/t5-ng-ta-gormez-tail-aug-lora", "/content/slam-asr-reimplement/tr/tsc/t0", "."]:
    if os.path.exists(os.path.join(cand, "slam_asr")):
        if cand not in sys.path:
            sys.path.insert(0, cand)
        break

try:
    from slam_asr.utils.metrics import compute_wer, compute_cer
except Exception:
    pass

def get_existing_metrics(folder_dir):
    summary_file = os.path.join(folder_dir, "eval_results_cv_summary.json")
    if os.path.exists(summary_file):
        try:
            with open(summary_file, "r", encoding="utf-8") as f:
                d = json.load(f)
                w, c = d.get("wer"), d.get("cer")
                if w is not None and c is not None:
                    return float(w), float(c)
        except Exception:
            pass
    eval_file = os.path.join(folder_dir, "eval_results_cv_generalization.jsonl")
    if not os.path.exists(eval_file) or os.path.getsize(eval_file) == 0:
        return None, None
    hyps, refs = [], []
    try:
        with open(eval_file, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                res = json.loads(line)
                if "metrics" in res or ("wer" in res and "hypothesis" not in res):
                    wer = res.get("metrics", {}).get("wer") or res.get("wer")
                    cer = res.get("metrics", {}).get("cer") or res.get("cer")
                    if wer is not None and cer is not None:
                        w = wer * 100 if wer <= 1.0 else wer
                        c = cer * 100 if cer <= 1.0 else cer
                        return float(w), float(c)
                hyps.append(res.get("hypothesis", ""))
                refs.append(res.get("reference", ""))
        if hyps and refs:
            wer = compute_wer(hyps, refs, language="tr")
            cer = compute_cer(hyps, refs, language="tr")
            w = wer * 100 if wer <= 1.0 else wer
            c = cer * 100 if cer <= 1.0 else cer
            try:
                with open(summary_file, "w", encoding="utf-8") as sf:
                    json.dump({"wer": round(w, 2), "cer": round(c, 2), "samples": len(hyps)}, sf, indent=2)
            except Exception:
                pass
            return float(w), float(c)
    except Exception as e:
        print(f"Hata ({os.path.basename(folder_dir)}): {e}")
    return None, None

print("
" + "=" * 95)
print("            TÜM MODELLERİN MOZILLA COMMON VOICE (TR) ÇAPRAZ TEST SONUÇLARI")
print("=" * 95)
print(f"{'No':<3} | {'Model Etiketi':<25} | {'Klasör Adı':<42} | {'WER (%)':<10} | {'CER (%)':<10}")
print("-" * 95)

for idx, ckpt in enumerate(all_tsc_checkpoints, 1):
    folder_dir = os.path.dirname(ckpt.rstrip("/"))
    folder_name = os.path.basename(folder_dir)
    model_tag = extract_model_tag(folder_name)
    w, c = get_existing_metrics(folder_dir)
    w_str = f"{w:.2f}%" if w is not None else "-"
    c_str = f"{c:.2f}%" if c is not None else "-"
    print(f"{idx:<3} | {model_tag:<25} | {folder_name[:42]:<42} | {w_str:<10} | {c_str:<10}")

print("=" * 95)